# 2. Preprocessing

Cleaning the dataset based on what I found in the inspection notebook. The cleaned file is saved at the end for the next notebooks.

In [1]:
import re
import pandas as pd

In [2]:
df = pd.read_csv('data/spam.csv', encoding='latin-1')
print(df.shape)

(5572, 5)


## Columns

Only `v1` and `v2` are needed.

In [3]:
df = df[['v1', 'v2']]
df = df.rename(columns={'v1': 'label', 'v2': 'message'})
df.isnull().sum()

label      0
message    0
dtype: int64

## Duplicates

In [4]:
print('Before:', df.shape)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print('After:', df.shape)

Before: (5572, 2)
After: (5169, 2)


## Labels

In [5]:
# ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df['label_num'].value_counts()

label_num
0    4516
1     653
Name: count, dtype: int64

## Cleaning the text

Steps: lowercase, replace everything that is not a letter or a number with a space, and remove extra spaces.

I'm keeping the numbers on purpose, because spam messages often have prices, phone numbers and short codes in them.

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # punctuation and strange characters
    text = re.sub(r'\s+', ' ', text)           # extra spaces
    return text.strip()

In [7]:
df['clean_message'] = df['message'].apply(clean_text)

spam_examples = df[df['label'] == 'spam'].sample(3, random_state=3)
ham_examples = df[df['label'] == 'ham'].sample(2, random_state=3)
examples = pd.concat([spam_examples, ham_examples])

for i, row in examples.iterrows():
    print(row['label'].upper())
    print('Original:', row['message'])
    print('Cleaned :', row['clean_message'])
    print()

SPAM
Original: Don't b floppy... b snappy & happy! Only gay chat service with photo upload call 08718730666 (10p/min). 2 stop our texts call 08712460324
Cleaned : don t b floppy b snappy happy only gay chat service with photo upload call 08718730666 10p min 2 stop our texts call 08712460324

SPAM
Original: TheMob> Check out our newest selection of content, Games, Tones, Gossip, babes and sport, Keep your mobile fit and funky text WAP to 82468
Cleaned : themob check out our newest selection of content games tones gossip babes and sport keep your mobile fit and funky text wap to 82468

SPAM
Original: 08714712388 between 10am-7pm Cost 10p
Cleaned : 08714712388 between 10am 7pm cost 10p

HAM
Original: HI HUN! IM NOT COMIN 2NITE-TELL EVERY1 IM SORRY 4 ME, HOPE U AVA GOODTIME!OLI RANG MELNITE IFINK IT MITE B SORTED,BUT IL EXPLAIN EVERYTHIN ON MON.L8RS.x
Cleaned : hi hun im not comin 2nite tell every1 im sorry 4 me hope u ava goodtime oli rang melnite ifink it mite b sorted but il explain eve

Some messages are only emoticons, so they might be empty after cleaning. Checking that.

In [8]:
empty = df[df['clean_message'] == '']
print(empty[['label', 'message']])

     label  message
3191   ham      :) 
4500   ham  :-) :-)


In [9]:
# nothing to learn from an empty message, so remove these
df = df[df['clean_message'] != '']
df = df.reset_index(drop=True)
print(df.shape)

(5167, 4)


## Duplicates after cleaning

Some messages only differed by a full stop or a space, so they are the same message now. I only noticed this when checking that the train and test sets didn't share messages, so it's better to remove them here.

In [10]:
print('Duplicates after cleaning:', df['clean_message'].duplicated().sum())

df = df.drop_duplicates(subset='clean_message')
df = df.reset_index(drop=True)
print(df.shape)

Duplicates after cleaning: 41
(5126, 4)


## What I'm not doing

- **Stopword removal:** words like "you", "your" and "to" are common in spam ("call now to claim your prize"), so removing them could hurt. I'll test this properly in the feature engineering notebook.
- **Stemming / lemmatization:** messages are short and full of slang and abbreviations ("u", "txt", "msg"), so I don't expect much from these.
- **Tokenization:** the vectorizers do this by themselves later.

In [11]:
df = df[['label', 'label_num', 'message', 'clean_message']]
df.to_csv('data/spam_clean.csv', index=False)
df.head()

,label,label_num,message,clean_message
0,ham,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,ham,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,ham,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,ham,0,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...


Saved as `data/spam_clean.csv`: 5,126 messages, with both the original and the cleaned text, and the 0/1 label.